In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import time
from collections import Counter
import urllib.request
import zipfile
import os

# Configure device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Store results for comparison
results = {}

Using device: cpu


In [10]:

def train_model(
    model,
    train_loader,
    val_loader,
    epochs=10,
    lr=0.001,
    task="regression",   # "regression" OR "classification"
    name="Model"
):
    """
    Training loop for stock prediction models.

    train_loader yields:
        x: (batch, seq_len, num_features)
        y: (batch, 1)

    task:
        "regression" → predicts price/return
        "classification" → predicts up/down
    """

    model = model.to(device)

    # ============================================
    # Loss function
    # ============================================
    if task == "regression":
        criterion = nn.MSELoss()
    elif task == "classification":
        criterion = nn.BCEWithLogitsLoss()
    else:
        raise ValueError("task must be 'regression' or 'classification'")

    optimizer = optim.Adam(model.parameters(), lr=lr)

    print(f"\nTraining {name} ({task}) for {epochs} epochs...")
    start_time = time.time()

    history = {
        'train_loss': [],
        'val_loss': []
    }

    # Optional metrics
    if task == "classification":
        history['train_acc'] = []
        history['val_acc'] = []

    for epoch in range(epochs):

        # ============================================
        # TRAIN
        # ============================================
        model.train()
        running_loss = 0.0

        correct = 0
        total = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device).float()

            optimizer.zero_grad()

            outputs = model(x).squeeze()

            loss = criterion(outputs, y)
            loss.backward()

            # Prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            running_loss += loss.item()

            # Classification accuracy
            if task == "classification":
                preds = (torch.sigmoid(outputs) > 0.5).long()
                correct += (preds == y.long()).sum().item()
                total += y.size(0)

        train_loss = running_loss / len(train_loader)
        history['train_loss'].append(train_loss)

        # ============================================
        # VALIDATION
        # ============================================
        model.eval()
        val_loss = 0.0

        correct = 0
        total = 0

        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                y = y.to(device).float()

                outputs = model(x).squeeze()
                loss = criterion(outputs, y)

                val_loss += loss.item()

                if task == "classification":
                    preds = (torch.sigmoid(outputs) > 0.5).long()
                    correct += (preds == y.long()).sum().item()
                    total += y.size(0)

        epoch_val_loss = val_loss / len(val_loader)
        history['val_loss'].append(epoch_val_loss)

        # ============================================
        # Logging
        # ============================================
        if task == "classification":
            train_acc = 100 * correct / total if total > 0 else 0
            val_acc = 100 * correct / total if total > 0 else 0

            history['train_acc'].append(train_acc)
            history['val_acc'].append(val_acc)

            print(f"Epoch [{epoch+1}/{epochs}] | "
                  f"Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {epoch_val_loss:.4f} | "
                  f"Val Acc: {val_acc:.2f}%")

        else:
            print(f"Epoch [{epoch+1}/{epochs}] | "
                  f"Train Loss: {train_loss:.6f} | "
                  f"Val Loss: {epoch_val_loss:.6f}")

    duration = time.time() - start_time
    print(f"{name} training completed in {duration:.2f}s")

    return history, duration

In [11]:
# Load the dataset
dataset = load_dataset("Adilbai/stock-dataset")
df = dataset["train"].to_pandas()
print(df.columns)
# Basic info
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Unique tickers: {df['Ticker'].nunique()}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/336M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/620095 [00:00<?, ? examples/s]

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends',
       'Stock Splits', 'Ticker', 'SMA_5', 'SMA_10', 'SMA_20', 'SMA_50',
       'EMA_12', 'EMA_26', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'RSI',
       'BB_Middle', 'BB_Upper', 'BB_Lower', 'BB_Width', 'BB_Position',
       'Volatility', 'Price_Change', 'Price_Change_5d', 'High_Low_Ratio',
       'Open_Close_Ratio', 'Volume_SMA', 'Volume_Ratio', 'Close_lag_1',
       'Close_lag_2', 'Close_lag_3', 'Close_lag_5', 'Close_lag_10',
       'Volume_lag_1', 'Volume_lag_2', 'Volume_lag_3', 'Volume_lag_5',
       'Volume_lag_10', 'Price_Change_lag_1', 'Price_Change_lag_2',
       'Price_Change_lag_3', 'Price_Change_lag_5', 'Price_Change_lag_10',
       'RSI_lag_1', 'RSI_lag_2', 'RSI_lag_3', 'RSI_lag_5', 'RSI_lag_10',
       'MACD_lag_1', 'MACD_lag_2', 'MACD_lag_3', 'MACD_lag_5', 'MACD_lag_10',
       'Volatility_lag_1', 'Volatility_lag_2', 'Volatility_lag_3',
       'Volatility_lag_5', 'Volatility_lag_10', 'Future_Return_1d',

# LSTM model

In [5]:

class LSTMStockModel(nn.Module):
    """
    LSTM for Stock Price Prediction (Time Series)

    Input:
        (batch, seq_len, num_features)
        Example features: [close, volume, RSI, MACD, etc.]

    Output:
        (batch, 1) -> next price OR return OR direction
    """

    def __init__(
        self,
        num_features,          # IMPORTANT: replaces vocab_size
        hidden_dim=128,
        output_dim=1,
        num_layers=2,
        dropout=0.2
    ):
        super(LSTMStockModel, self).__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # ============================================
        # LSTM Layer (NO embedding)
        # ============================================
        self.lstm = nn.LSTM(
            input_size=num_features,   # e.g. 5–50 features
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False  # IMPORTANT: no future leakage
        )

        # ============================================
        # Fully Connected Head
        # ============================================
        self.fc = nn.Linear(hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        x: (batch, seq_len, num_features)
        """

        # LSTM output
        out, (hidden, cell) = self.lstm(x)

        # Option 1 (recommended): last timestep output
        final_output = out[:, -1, :]   # (batch, hidden_dim)

        # Option 2 (also valid): hidden[-1]
        # final_output = hidden[-1]

        final_output = self.dropout(final_output)

        return self.fc(final_output)

In [ ]:
# =========================
# 1. Imports
# =========================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

# =========================
# 2. Load dataset
# =========================
dataset = load_dataset("Adilbai/stock-dataset")
df = dataset["train"].to_pandas()

print(df.columns)
print(df.shape)

# =========================
# 3. Sort data (VERY IMPORTANT)
# =========================
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"])
one_year_df = (
    df.groupby("Ticker")
      .apply(lambda x: x[x["Date"] >= x["Date"].max() - pd.DateOffset(years=1)])
      .reset_index(drop=True)
)

print(one_year_df.shape)

# =========================
# 4. Choose features + target
# =========================
features = ["Open", "High", "Low", "Close", "Volume", "SMA_5"]
target = "Future_Return_1d"

df = one_year_df.dropna(subset=features + [target])

# =========================
# 5. Create sequences per ticker
# =========================
SEQ_LEN = 30

def create_sequences(df, features, target, seq_len):
    X, y = [], []

    for ticker in df["Ticker"].unique():
        stock = df[df["Ticker"] == ticker]

        data = stock[features].values
        labels = stock[target].values

        for i in range(len(stock) - seq_len):
            X.append(data[i:i+seq_len])
            y.append(labels[i+seq_len])

    return np.array(X), np.array(y)

X, y = create_sequences(df, features, target, SEQ_LEN)

print("Raw X shape:", X.shape)
print("Raw y shape:", y.shape)

# =========================
# 6. Train/test split (time-safe)
# =========================
split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# =========================
# 7. Normalize (fit ONLY on train)
# =========================
scaler = StandardScaler()

n_samples, seq_len, n_features = X_train.shape

X_train_reshaped = X_train.reshape(-1, n_features)
X_test_reshaped = X_test.reshape(-1, n_features)

X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_test_scaled = scaler.transform(X_test_reshaped)

X_train = X_train_scaled.reshape(n_samples, seq_len, n_features)
X_test = X_test_scaled.reshape(X_test.shape[0], seq_len, n_features)

# =========================
# 8. PyTorch Dataset
# =========================
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = StockDataset(X_train, y_train)
test_ds = StockDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

# =========================
# 9. LSTM Model
# =========================
class LSTMStockModel(nn.Module):
    def __init__(self, num_features, hidden_dim=128, output_dim=1, num_layers=2, dropout=0.2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]          # last timestep
        out = self.dropout(out)
        return self.fc(out)

# =========================
# 10. Init model
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMStockModel(num_features=n_features).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# =========================
# 11. Training loop
# =========================
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)

        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss / len(train_loader):.6f}")

# =========================
# 12. Evaluation
# =========================
model.eval()
test_loss = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        test_loss += loss.item()

print("Test Loss:", test_loss / len(test_loader))

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends',
       'Stock Splits', 'Ticker', 'SMA_5', 'SMA_10', 'SMA_20', 'SMA_50',
       'EMA_12', 'EMA_26', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'RSI',
       'BB_Middle', 'BB_Upper', 'BB_Lower', 'BB_Width', 'BB_Position',
       'Volatility', 'Price_Change', 'Price_Change_5d', 'High_Low_Ratio',
       'Open_Close_Ratio', 'Volume_SMA', 'Volume_Ratio', 'Close_lag_1',
       'Close_lag_2', 'Close_lag_3', 'Close_lag_5', 'Close_lag_10',
       'Volume_lag_1', 'Volume_lag_2', 'Volume_lag_3', 'Volume_lag_5',
       'Volume_lag_10', 'Price_Change_lag_1', 'Price_Change_lag_2',
       'Price_Change_lag_3', 'Price_Change_lag_5', 'Price_Change_lag_10',
       'RSI_lag_1', 'RSI_lag_2', 'RSI_lag_3', 'RSI_lag_5', 'RSI_lag_10',
       'MACD_lag_1', 'MACD_lag_2', 'MACD_lag_3', 'MACD_lag_5', 'MACD_lag_10',
       'Volatility_lag_1', 'Volatility_lag_2', 'Volatility_lag_3',
       'Volatility_lag_5', 'Volatility_lag_10', 'Future_Return_1d',

/tmp/ipykernel_19925/4000236714.py:24: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["Date"] = pd.to_datetime(df["Date"])
/tmp/ipykernel_19925/4000236714.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x[x["Date"] >= x["Date"].max() - pd.DateOffset(years=1)])


(126236, 73)
Raw X shape: (111146, 30, 6)
Raw y shape: (111146,)
Epoch 1/10, Loss: 0.000484
